In [0]:
select * from investment_pyspark.silver.holdings_clean;

In [0]:
select * from investment_pyspark.silver.dividens_cleaned;

In [0]:
%sql
with dividend_by_stock as (
select Symbol,sum(Total_dividend) as total_dividend_income,count(*) as dividend_payment
from investment_pyspark.silver.dividens_cleaned
group by Symbol)
select h.Instrument,h.Qty,h.Invested_value,h.Current_value,h.Unrealized_pnl,
coalesce(d.total_dividend_income,0) as total_dividend_income,
coalesce(d.dividend_payment,0) as dividend_payment
from investment_pyspark.silver.holdings_clean h
left join dividend_by_stock d
on h.Instrument=d.Symbol;
--order by h.Invested_value desc,d.total_dividend_income desc;

### Total Return

In [0]:
create or replace table investment_pyspark.gold.total_return as (
with dividend_by_stock as (
select Symbol,
sum(Total_dividend) as total_dividend,
count(*) as dividend_payment
from investment_pyspark.silver.dividens_cleaned
group by Symbol)
select h.Instrument,h.Invested_value,h.Current_value,h.Unrealized_pnl,
coalesce(d.total_dividend,0) as total_dividend_income,
h.Unrealized_pnl + coalesce(d.total_dividend,0) as total_pnl,
coalesce(d.dividend_payment,0) as dividend_payment
from investment_pyspark.silver.holdings_clean h
left join dividend_by_stock d 
on h.Instrument=d.Symbol)

In [0]:
select * from investment_pyspark.gold.total_return
order by total_pnl desc;

In [0]:
select sum(total_pnl),sum(Unrealized_pnl
) from investment_pyspark.gold.total_return;

## transaction summary

In [0]:
create or replace table investment_pyspark.gold.transaction_summary as
(
select symbol,isin,exchange,segment,trade_type,sum(quantity) as bought_qty,sum(price*quantity) as bought_price
from investment_pyspark.silver.tradebook_cleaned
group by symbol,isin,exchange,segment,trade_type)

In [0]:
select * from investment_pyspark.gold.transaction_summary;

In [0]:
select sum(total_value) as total_invested_value from investment_pyspark.gold.transaction_summary;

##Transaction and Holding summary

In [0]:
%skip
select h.asset_class,t.trade_type,t.segment,h.Instrument,t.isin,h.Qty,sum(t.quantity) as Bought_Qty,h.Avg_cost,h.Current_price,h.Invested_value,
sum(t.quantity*t.price) as bought_value,h.Current_value
from investment_pyspark.silver.holdings_clean h
join investment_pyspark.silver.tradebook_cleaned t on h.Instrument=t.symbol
group by h.Instrument,t.isin,h.asset_class,h.Qty,h.Avg_cost,h.Current_price,h.Invested_value,h.Current_value,
t.segment,t.trade_type